# Domain-Specific RAG Chatbot

### Internship Project 6

**Objective:**
Build a chatbot that answers questions from a PDF document using Retrieval-Augmented Generation (RAG).

### Technologies Used

- Python
- Jupyter Notebook
- LangChain
- ChromaDB
- Sentence Transformers
- OpenAI / Anthropic
- Streamlit

In [1]:
import sys
print(sys.version)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
!pip install -U \
streamlit \
langchain \
langchain-community \
langchain-huggingface \
chromadb \
sentence-transformers \
pypdf \
python-dotenv \
openai

  Using cached jiter-0.16.0-cp311-cp311-win_amd64.whl.metadata (5.3 kB)
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB 93.5 kB/s eta 0:01:52
   ---------------------------------------- 0.0/10.4 MB 93.5 kB/s eta 0:01:52
   ---------------------------------------- 0.0/10.4 MB 93.5 kB/s eta 0:01:52
   ---------------------------------------- 0.0/10.4 MB 93.5 kB/s eta 0:01:52
   ---------------------------------------- 0.0/10.4 MB 93.5 kB/s eta 0:01:52
   ---------------------------------------- 0.0/10.4 MB 93.5 kB/s eta 0:01:52
   -------------------------------------

ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\MANISHA\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\Users\MANISHA\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\MANISHA\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\MANISHA\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\MANISHA\AppData\Local\Programs\Python\Python311\Lib\http\client.py", line 473, in read
    s = se

In [3]:
import streamlit
import langchain
import chromadb
import sentence_transformers
import pypdf

print("✅ All libraries installed successfully!")

✅ All libraries installed successfully!


In [6]:
!pip install langchain-chroma


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# PDF Loader
from langchain_community.document_loaders import PyPDFLoader

# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding Model
from langchain_huggingface import HuggingFaceEmbeddings

# Vector Database
from langchain_chroma import Chroma

print("✅ All libraries imported successfully!")



✅ All libraries imported successfully!


In [10]:
loader = PyPDFLoader("../data/textbook.pdf")

documents = loader.load()

print("Total Pages:", len(documents))

Total Pages: 117


In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 285


In [12]:
print(chunks[0].page_content)

Machine Learning
CSC_2S004_EP (previously: CSE204)
Jesse Read
jesse.read@polytechnique.edu
Version: May 11, 2025
These Lecture Notes are still a work in progress andwill be updated and adapted through-
out the course (expect updates each week). There is approximately one chapter per top-
ic/week. These notes are not intended as a complete reference of all course material; they
are designed to be read to complement (not to substitute) lectures. Neither are these notes
uniquely required for a full understanding of course material. Machine Learning is by now a
very well-covered topic, you will find many alternative (often, open-source) references.
The main objective of these notes is to introduce and develop theoretical concepts which are
presented in the lectures. Practical machine learning is also an important component of the
course. Practical aspects will be discussed in lectures, but mainly covered in the Tutorial/Lab
sessions.


In [13]:
print(chunks[0].metadata)

{'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-05-11T16:58:42+02:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2025-05-11T16:58:42+02:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2025/dev/Debian) kpathsea version 6.4.0/dev', 'source': '../data/textbook.pdf', 'total_pages': 117, 'page': 0, 'page_label': '1'}


In [14]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model loaded successfully!


In [15]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="../chroma_db"
)

print("✅ Chroma Database created successfully!")

✅ Chroma Database created successfully!


In [16]:
import os

if os.path.exists("../chroma_db"):
    print("✅ Chroma Database folder exists.")
else:
    print("❌ Database folder not found.")

✅ Chroma Database folder exists.


In [18]:
print("Total Chunks Stored:", vector_db._collection.count())

Total Chunks Stored: 570


In [19]:
question = "What is supervised learning?"

print(question)

What is supervised learning?


In [84]:
retrieved_docs = vector_db.similarity_search(
    question,
    k=5
)

print("Retrieved", len(retrieved_docs), "chunks")

Retrieved 5 chunks


In [35]:
for i, doc in enumerate(retrieved_docs, start=1):
    print("=" * 80)
    print(f"Chunk {i}")
    print(doc.page_content)

Chunk 1
induced climate change, because this did not happen in the past, and we have no evidence of this ever
happening2. In other words, a potential failure to model uncertainty about the future.
4.4 Bias-Variance Decomposition
We have so far been discussing uncertainty regarding our prediction, under a fixed model (producing that
prediction). Yet, what about the uncertainty regarding which model to start with? We need to think
about what wedo and do notknow (i.e., our uncertainty). Wedo haveℓ (let’s suppose squared error3),
we do have data{xi,yi}n
i=1. But, wedo nothave the trueP from which the data was sourced/generated.
Our uncertainty is aroundY and ˆF. ˆF? But isn’t ˆf determined deterministically, when we train our
model? Indeed, ˆf is fit on the training data which is sourced from an unknown distribution; i.e., a random
variable; so the modelling decision inherits this uncertainty, and becomes a random variable itself;ˆF.
Chunk 2
induced climate change, because this did not hap

In [85]:
print("\nSources:")

pages = sorted(set(doc.metadata["page"] + 1 for doc in retrieved_docs))

for page in pages:
    print(f"Page {page}")


Sources:
Page 34
Page 36
Page 78


In [74]:
context = ""

for i, doc in enumerate(retrieved_docs):
    context += f"\n----- Chunk {i+1} -----\n"
    context += doc.page_content
    context += "\n"

print(context[:3000])


----- Chunk 1 -----
induced climate change, because this did not happen in the past, and we have no evidence of this ever
happening2. In other words, a potential failure to model uncertainty about the future.
4.4 Bias-Variance Decomposition
We have so far been discussing uncertainty regarding our prediction, under a fixed model (producing that
prediction). Yet, what about the uncertainty regarding which model to start with? We need to think
about what wedo and do notknow (i.e., our uncertainty). Wedo haveℓ (let’s suppose squared error3),
we do have data{xi,yi}n
i=1. But, wedo nothave the trueP from which the data was sourced/generated.
Our uncertainty is aroundY and ˆF. ˆF? But isn’t ˆf determined deterministically, when we train our
model? Indeed, ˆf is fit on the training data which is sourced from an unknown distribution; i.e., a random
variable; so the modelling decision inherits this uncertainty, and becomes a random variable itself;ˆF.

----- Chunk 2 -----
induced climate change

In [75]:
prompt = f"""
You are a Machine Learning tutor.

Answer the question ONLY using the information given below.

If the answer exists in the context, explain it in simple words.

Do NOT say "I don't have enough information" unless the answer is completely missing.

Context:
{context}

Question:
{question}

Answer:
"""

In [39]:
! pip install ollama


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [76]:
response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response["message"]["content"]

print(answer)

There is no information about the FIFA World Cup 2022 or any other sports event in the provided context. The text appears to be related to machine learning, climate change, and model training, but it does not contain any information about sports events.


In [97]:
print("\nSources Used:\n")

for doc in retrieved_docs:
    print(f"Page {doc.metadata['page'] + 1}")


Sources Used:

Page 36
Page 36
Page 34
Page 34
Page 78


In [92]:
def ask_question(question):
    # Retrieve relevant documents
    retrieved_docs = vector_db.similarity_search(question, k=5)

    # Build context
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # Create prompt
    prompt = f"""
You are a helpful AI assistant.

Answer ONLY using the context provided below.

Keep your answer short (2-3 sentences).

If the answer is not available in the context, reply exactly:

I don't have enough information from the provided documents.

Context:
{context}

Question:
{question}

Answer:
"""

    # Get response from Ollama
    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    # Print answer
    answer = response["message"]["content"]

    print("=" * 80)
    print(f"Question: {question}\n")
    print("Answer:")
    print(answer)

    # Print unique source pages
    print("\nSources:")
    pages = sorted(set(doc.metadata["page"] + 1 for doc in retrieved_docs))

    for page in pages:
        print(f"Page {page}")

    print("=" * 80)

In [93]:
ask_question("What is supervised machine learning?")

Question: What is supervised machine learning?

Answer:
Supervised machine learning involves building a model to assign labels to instances in a training set based on available labels. This approach enables tasks such as classification and regression analysis. Supervised machine learning aims to minimize expected error by optimizing a loss function.

Sources:
Page 11
Page 15
Page 16


In [94]:
ask_question("What is unsupervised machine learning?")

Question: What is unsupervised machine learning?

Answer:
Unsupervised machine learning involves building a model to assign labels to instances without prior labeled training data, often used for tasks like clustering analysis or representation learning.

Sources:
Page 16
Page 73
Page 84


In [95]:
ask_question("Who won the FIFA World Cup 2022?")

Question: Who won the FIFA World Cup 2022?

Answer:
I don't have enough information from the provided documents.

Sources:
Page 34
Page 36
Page 78
